# Задание 1: Induction Heads DIY

Двуслойный attention-only трансформер без layernorm, одна голова на каждом слое.

- Словарь: `V = {A, B, C}`, размерность эмбеддинга `d = 7`, размерность головы `dh = 3`
- Абсолютные позиционные эмбеддинги: `p(t) = [0, 0, 0, 0, 0, 0, t]`, `t ∈ {1, 2, ...}`
- Токены кодируются one-hot в первых трёх координатах: `A=[1,0,0,...]`, `B=[0,1,0,...]`, `C=[0,0,1,...]`
- Слой 1: **previous-token head** — копирует предыдущий токен (строгая каузальная маска)
- Слой 2: **induction head** — от токена `x` смотрит на токен `y`, перед которым стоит `x`

In [4]:
import numpy as np

# -- константы ------------------------------------------------------------------
D   = 7   # размерность эмбеддинга
DH  = 3   # размерность головы
V   = 3   # размер словаря: A=0, B=1, C=2

TOKEN = {'A': 0, 'B': 1, 'C': 2}
IDX2TOK = {0: 'A', 1: 'B', 2: 'C'}

In [5]:
# -- кодирование входной последовательности -------------------------------------

def encode(sequence: list[str]) -> np.ndarray:
    """
    Принимает список токенов, возвращает матрицу эмбеддингов (T, D).

    Каждый эмбеддинг = токен_one_hot (dim 0-2) + нули (dim 3-5) + позиция (dim 6).
    Позиция 1-индексирована: первый токен имеет t=1.
    """
    T = len(sequence)
    X = np.zeros((T, D))  # (T, D)
    for t, tok in enumerate(sequence):
        X[t, TOKEN[tok]] = 1.0        # one-hot токена в dim 0-2
        X[t, 6]          = float(t + 1)  # абсолютная позиция в dim 6
    return X

# санити-чек: A, B, A
print(encode(['A', 'B', 'A']))

[[1. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 2.]
 [1. 0. 0. 0. 0. 0. 3.]]


In [6]:

# -- механизм attention ---------------------------------------------------------

def strict_causal_mask(T: int) -> np.ndarray:
    """
    Строгая каузальная маска (T, T):
    mask[i, j] = True  если j < i (допустимое внимание),
    mask[i, j] = False если j >= i (включая главную диагональ — зануляем).
    """
    return np.tril(np.ones((T, T), dtype=bool), k=-1)  # строго ниже диагонали


from scipy.special import softmax
def attention(X: np.ndarray | list[list[float]],
              Wq: np.ndarray | list[list[float]], Wk: np.ndarray | list[list[float]],
              Wv: np.ndarray | list[list[float]], Wo: np.ndarray | list[list[float]],
              verbose: bool = False,
              layer_name: str = "") -> tuple[np.ndarray, np.ndarray]:
    """
    Один слой attention (без layernorm, без MLP).

    Параметры
    ---------
    X  : (T, D)  — входные эмбеддинги
    Wq : (D, DH) — матрица запросов
    Wk : (D, DH) — матрица ключей
    Wv : (D, DH) — матрица значений
    Wo : (DH, D) — матрица проекции выхода

    Возвращает
    ----------
    out   : (T, D) — X + residual
    attn  : (T, T) — карта внимания (после softmax)
    """
    X, Wq, Wk, Wv, Wo = np.array(X), np.array(Wq), np.array(Wk), np.array(Wv), np.array(Wo)
    T = X.shape[0]

    Q = X @ Wq  # (T, DH)
    K = X @ Wk  # (T, DH)
    V_ = X @ Wv  # (T, DH)

    # логиты: (T, T)
    scores = Q @ K.T / np.sqrt(DH)  # (T, T)

    # строгая каузальная маска
    mask = strict_causal_mask(T)
    scores_masked = np.where(mask, scores, -np.inf)  # маскируем недопустимые позиции
    scores_masked[0] = 0  # избегаем предупреждений
    attn = np.array(softmax(scores_masked, axis=1))  # (T, T)
    # первый токен: нет доступных позиций -> нулевая строка
    attn[0] = 0.0

    head_out = attn @ V_       # (T, DH)
    projected = head_out @ Wo  # (T, D)

    out = X + projected  # residual connection

    if verbose:
        print(f"\n{'='*60}")
        print(f"Layer: {layer_name}")
        print(f"Q =\n{np.round(Q, 3)}")
        print(f"K =\n{np.round(K, 3)}")
        print(f"Raw scores =\n{np.round(scores, 3)}")
        print(f"Attention map (после softmax) =\n{np.round(attn, 3)}")
        print(f"V =\n{np.round(V_, 3)}")
        print(f"head_out =\n{np.round(head_out, 3)}")
        print(f"projected =\n{np.round(projected, 3)}")
        print(f"out (X + projected) =\n{np.round(out, 3)}")

    return out, attn

print(strict_causal_mask(3))

print(attention([[0, 1], [1, 0]], [[1, 1], [0, 1]], [[1, 0], [0, 1]], [[1, 0], [0, 1]],
                [[52, 42], [67, 88]]))

[[False False False]
 [ True False False]
 [ True  True False]]
(array([[ 0.,  1.],
       [68., 88.]]), array([[0., 0.],
       [1., 0.]]))


In [30]:
We = np.hstack(
    [np.eye(3, dtype=np.float32), np.zeros((DH, D - 3), dtype=np.float32)]
)  # (3, 7)

# -- матрица разэмбеддинга ------------------------------------------------------
#
# Проецирует последний эмбеддинг размерности D -> V (логиты по словарю).
# Простейший вариант: извлечь dim 0-2 (one-hot токена).

Wu = We.T

# -- веса модели ----------------------------------------------------------------
#
#
# Слой 1: previous-token head
# Цель: каждый токен на позиции t должен записать в свободные dim 3-5
# представление токена с позиции t-1.
#
# Механика:
#   Q использует позиционный dim (dim 6), чтобы спросить «кто стоит ровно до меня?»
#   K отвечает на основе позиции.
#   V копирует one-hot токена (dim 0-2) в выходной вектор.
#   O записывает скопированный токен в свободные dim 3-5.

Wq1 = np.array(
    [[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [1, 0, 0]],
    dtype=np.float32,
)  # (7, 3)
Wk1 = Wq1.copy() * 5  # (7, 3)
Wv1 = We.T  # (7, 3)
Wo1 = np.vstack(
    [
        np.zeros((3, DH), dtype=np.float32),
        np.eye(3, dtype=np.float32),
        np.zeros((1, DH), dtype=np.float32),
    ]
).T  # (7, 3)

# -- Слой 2: induction head -----------------------------------------------------
#
# Цель: токен x на позиции t должен найти такой токен y,
# перед которым встречался x (т.е. dim 3-5 у y хранят one-hot x).
# Затем предсказать токен после y.
#
# Механика:
#   Q использует dim 0-2 (текущий токен x).
#   K использует dim 3-5 (скопированный предыдущий токен из слоя 1).
#   Совпадение Q*K^T высокое <-> перед y стоял токен x.
#   V копирует one-hot самого токена y (dim 0-2).
#   O записывает это в логиты (через unembedding).

Wq2 = We.T  # (7, 3)
Wk2 = 5 * np.vstack(
    [
        np.zeros((3, DH), dtype=np.float32),
        np.eye(3, dtype=np.float32),
        np.zeros((1, DH), dtype=np.float32),
    ] 
)  # (7, 3)
Wv2 = We.T  # (7, 3)
Wo2 = We.copy() * 5  # (3, 7)

In [31]:
# -- форвард-пасс --------------------------------------------------------------

def forward(sequence: list[str], verbose: bool = False) -> np.ndarray:
    """
    Полный форвард трансформера.

    Возвращает вероятности следующего токена для ПОСЛЕДНЕЙ позиции (V,).
    """
    X = encode(sequence)  # (T, D)

    if verbose:
        print("Входная матрица эмбеддингов X:")
        print(np.round(X, 3))

    # слой 1: previous-token head
    X1, attn1 = attention(X, Wq1, Wk1, Wv1, Wo1,
                          verbose=verbose, layer_name="Layer 1 (prev-token head)")

    # слой 2: induction head
    X2, attn2 = attention(X1, Wq2, Wk2, Wv2, Wo2,
                          verbose=verbose, layer_name="Layer 2 (induction head)")

    # логиты и вероятности для последнего токена
    logits = X2[-1] @ Wu  # (V,)
    probs = softmax(logits)  # (V,)

    if verbose:
        print(f"\nЛогиты (последняя позиция): {np.round(logits, 3)}")
        print(f"Вероятности: { {IDX2TOK[i]: round(float(probs[i]), 4) for i in range(V)} }")

    return probs


In [32]:
# -- тестовые последовательности -----------------------------------------------

test_cases = [
    (['A', 'B', 'A'],          'B'),
    (['C', 'A', 'C'],          'A'),
    (['B', 'C', 'C', 'A', 'B'], 'C'),
    (['A', 'A', 'A', 'A'],     'A'),
]

print(f"{'Последовательность':<30} {'Ожидается':<12} {'P(правильный)':<15}")
print("-" * 57)
for seq, expected in test_cases:
    probs = forward(seq, verbose=False)
    p_correct = probs[TOKEN[expected]]
    print(f"{', '.join(seq):<30} {expected:<12} {p_correct:.6f}")

Последовательность             Ожидается    P(правильный)  
---------------------------------------------------------
A, B, A                        B            0.961693
C, A, C                        A            0.961693
B, C, C, A, B                  C            0.951212
A, A, A, A                     A            0.995067


In [33]:
# -- подробный разбор примера A, B, A -----------------------------------------
# (раскомментировать после подбора весов)

_ = forward(['A', 'B', 'A'], verbose=True)

Входная матрица эмбеддингов X:
[[1. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 2.]
 [1. 0. 0. 0. 0. 0. 3.]]

Layer: Layer 1 (prev-token head)
Q =
[[1. 0. 0.]
 [2. 0. 0.]
 [3. 0. 0.]]
K =
[[ 5.  0.  0.]
 [10.  0.  0.]
 [15.  0.  0.]]
Raw scores =
[[ 2.887  5.774  8.66 ]
 [ 5.774 11.547 17.321]
 [ 8.66  17.321 25.981]]
Attention map (после softmax) =
[[0. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]]
V =
[[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]]
head_out =
[[0. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]]
projected =
[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]]
out (X + projected) =
[[1. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 1. 0. 0. 2.]
 [1. 0. 0. 0. 1. 0. 3.]]

Layer: Layer 2 (induction head)
Q =
[[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]]
K =
[[0.000e+00 0.000e+00 0.000e+00]
 [5.000e+00 0.000e+00 0.000e+00]
 [1.000e-03 4.999e+00 0.000e+00]]
Raw scores =
[[0.000e+00 2.887e+00 1.000e-03]
 [0.000e+00 0.000e+00 2.886e+00]
 [0.000e+00 2.887e+00 1.000e-03]]
Attention map (после softmax) =
[[0.    0.    0.   ]
 [1